# Invoice reader: from images to a table

**dabeer.ai · AI for Beginners · Python and AI APIs**

In this notebook an AI model reads three sample invoices, returns the details as JSON, and your code checks every reply before it goes into a table. The rule from the lesson: **ask for JSON, check it with code, and if it's wrong, ask again.**

**Before you start:** your API key must be in Colab Secrets (the key icon in the left sidebar), named `GEMINI_API_KEY`, `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`, with **Notebook access** on. Have the three invoice images from the starter folder ready on your computer.

### No paid API key? Use Gemini's free key

This notebook starts with `PROVIDER = "gemini"`, because Google AI Studio gives you a free Gemini API key, with no payment method needed.

1. Go to https://aistudio.google.com/apikey and sign in with your Google account.
2. Click **Create API key**, and copy the key.
3. In Colab, click the **key icon** in the left sidebar, add a secret named `GEMINI_API_KEY`, paste the key as the value, and switch on **Notebook access**.

Two things to know about the free key:

- It has rate limits. If you see a "429" or "resource exhausted" error, wait a minute and run the cell again.
- On the free tier, Google may use what you send, and the replies, to improve its products, and people may read them. So use only fictional or practice data, like the samples here. Never real customer, company or personal data.

Have a paid Anthropic or OpenAI key instead? Set `PROVIDER` to `"anthropic"` or `"openai"` in the settings cell.

## Step 1: Install the libraries

In [ ]:
!pip install -q google-genai anthropic openai

## Step 2: Settings

In [ ]:
# ---- Settings: change these ----

PROVIDER = "gemini"   # "gemini" (free key), "anthropic" or "openai"

# Model names change. Check the provider's model list for current names.
# The model must accept images.
MODELS = {
    "gemini": "gemini-3.1-flash-lite",   # model names change: pick a current one from the AI Studio list
    "anthropic": "claude-sonnet-5-5",
    "openai": "gpt-5-mini",
}
MODEL = MODELS[PROVIDER]

MAX_TOKENS = 4000   # the most tokens one reply may use
MAX_TRIES = 3       # how many times to ask again if the reply fails the format check
TOLERANCE = 1.0     # rupees of rounding difference we accept when checking totals

## Step 3: Connect to the provider

In [ ]:
# Reads your API key from Colab Secrets, so it never appears in this notebook.
from google.colab import userdata

KEY_NAMES = {"gemini": "GEMINI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}

try:
    api_key = userdata.get(KEY_NAMES[PROVIDER])
except Exception as error:
    raise RuntimeError(
        f"Couldn't read the secret {KEY_NAMES[PROVIDER]}. Open the key icon in the left sidebar, "
        "check the name is exactly right, and switch on 'Notebook access'. "
        f"Details: {error}"
    ) from None

if PROVIDER == "gemini":
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=api_key)
elif PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic(api_key=api_key)
elif PROVIDER == "openai":
    import openai
    client = openai.OpenAI(api_key=api_key)
else:
    raise ValueError('PROVIDER must be "gemini", "anthropic" or "openai"')

print("Ready. Provider:", PROVIDER, "| Model:", MODEL)

## Step 4: Upload the invoices

Run the cell and choose the three PNG files from the starter `invoices` folder (you can select them all at once). If they're already uploaded, the cell skips this.

In [ ]:
import glob
import os
from IPython.display import Image, display

INVOICE_FOLDER = "invoices"
os.makedirs(INVOICE_FOLDER, exist_ok=True)

def find_invoices():
    paths = []
    for pattern in ("*.png", "*.jpg", "*.jpeg"):
        paths += glob.glob(os.path.join(INVOICE_FOLDER, pattern))
    return sorted(paths)

if not find_invoices():
    from google.colab import files
    print("Choose the invoice images to upload.")
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(INVOICE_FOLDER, name), "wb") as f:
            f.write(data)

invoice_paths = find_invoices()
print(f"Found {len(invoice_paths)} invoices:", invoice_paths)
display(Image(invoice_paths[0], width=450))   # show the first one

## Step 5: The prompt

Be exact: the keys, the date format, numbers without currency signs, and one small example of the shape. Notice the rule **copy every number exactly as printed**. We want the AI to read, not to fix. Fixing is your code's job.

In [ ]:
EXTRACTION_PROMPT = """Read this invoice image. Return ONLY a JSON object, with no other text and no code fences.
Use exactly these keys and this shape:
{
  "vendor": "name of the company that sent the invoice",
  "invoice_number": "as printed",
  "invoice_date": "YYYY-MM-DD",
  "line_items": [
    {"description": "text", "quantity": 2, "unit_price": 150.0, "amount": 300.0}
  ],
  "subtotal": 300.0,
  "tax_percent": 18.0,
  "tax_amount": 54.0,
  "total": 354.0
}
Rules:
- All amounts are plain numbers: no currency symbol, no commas, no words.
- If the tax is split into parts (for example CGST and SGST), add the parts together.
- Dates written like 03/10/2026 are Indian format, day first.
- Copy every number exactly as printed, even if it looks wrong. Do not correct anything."""

## Step 6: Send one image to the model

Anthropic and OpenAI take the image as base64 text; Gemini takes the image bytes as they are. Each provider wraps the image in a slightly different shape, so there's one branch for each.

In [ ]:
import base64

def image_message(path, prompt):
    """Build one user message that holds an image and a prompt."""
    media_type = "image/png" if path.lower().endswith(".png") else "image/jpeg"
    with open(path, "rb") as f:
        image_bytes = f.read()
    image_b64 = base64.standard_b64encode(image_bytes).decode("utf-8")
    if PROVIDER == "gemini":
        content = [
            {"inline_data": {"mime_type": media_type, "data": image_bytes}},
            {"text": prompt},
        ]
    elif PROVIDER == "anthropic":
        content = [
            {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": image_b64}},
            {"type": "text", "text": prompt},
        ]
    else:
        content = [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{media_type};base64,{image_b64}"}},
        ]
    return {"role": "user", "content": content}

def send_to_model(messages):
    """Returns the reply text."""
    if PROVIDER == "gemini":
        # Gemini calls the assistant "model", and wants each message as a list of "parts"
        contents = []
        for m in messages:
            parts = m["content"] if isinstance(m["content"], list) else [{"text": m["content"]}]
            contents.append({"role": "model" if m["role"] == "assistant" else "user", "parts": parts})
        response = client.models.generate_content(
            model=MODEL, contents=contents, config=types.GenerateContentConfig(max_output_tokens=MAX_TOKENS))
        return response.text or ""
    elif PROVIDER == "anthropic":
        response = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, messages=messages)
        return "".join(block.text for block in response.content if block.type == "text")
    else:
        response = client.chat.completions.create(model=MODEL, max_completion_tokens=MAX_TOKENS, messages=messages)
        return response.choices[0].message.content or ""

# Try it on the first invoice and look at the raw reply.
first_reply = send_to_model([image_message(invoice_paths[0], EXTRACTION_PROMPT)])
print(first_reply)

## Step 7: Check the format

`check_format` returns a list of problems. An empty list means the reply passed. It checks three things: is it valid JSON, are all the keys there, and are the values the right kind (numbers are numbers, the date is a real date).

In [ ]:
import json
import re
from datetime import datetime

REQUIRED_KEYS = ["vendor", "invoice_number", "invoice_date", "line_items",
                 "subtotal", "tax_percent", "tax_amount", "total"]
NUMBER_KEYS = ["subtotal", "tax_percent", "tax_amount", "total"]
ITEM_NUMBER_KEYS = ["quantity", "unit_price", "amount"]

def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)

def parse_reply(reply):
    """Turn the reply text into a Python dictionary. Returns (data, problems)."""
    text = reply.strip()
    text = re.sub(r"^```(json)?|```$", "", text).strip()   # remove code fences if the model added them
    try:
        data = json.loads(text)
    except json.JSONDecodeError as error:
        return None, [f"not valid JSON ({error})"]
    if not isinstance(data, dict):
        return None, ["the JSON is not an object with keys"]
    return data, check_format(data)

def check_format(data):
    problems = []
    for key in REQUIRED_KEYS:
        if key not in data:
            problems.append(f"missing key: {key}")
    for key in NUMBER_KEYS:
        if key in data and not is_number(data[key]):
            problems.append(f"{key} is not a number: {data[key]!r}")
    if "invoice_date" in data:
        try:
            datetime.strptime(str(data["invoice_date"]), "%Y-%m-%d")
        except ValueError:
            problems.append(f"invoice_date is not YYYY-MM-DD: {data['invoice_date']!r}")
    items = data.get("line_items")
    if not isinstance(items, list) or not items:
        problems.append("line_items must be a non-empty list")
    else:
        for n, item in enumerate(items, start=1):
            for key in ITEM_NUMBER_KEYS:
                if not is_number(item.get(key)):
                    problems.append(f"line {n}: {key} is not a number: {item.get(key)!r}")
    return problems

# Test the checks on a deliberately bad reply, then on the real one from Step 6.
print(parse_reply('{"vendor": "Test", "total": "twelve thousand rupees"}')[1])
print(parse_reply(first_reply)[1] or "First reply passed the format check.")

## Step 8: Ask again, with feedback

If the reply fails, we send the problems back and ask again, up to `MAX_TRIES` times. If it still fails, the invoice is flagged for a person. We don't stop the whole run.

In [ ]:
def read_invoice(path):
    """Returns (data, problems). problems is empty if the reply passed."""
    messages = [image_message(path, EXTRACTION_PROMPT)]
    problems = ["no reply"]
    data = None
    for attempt in range(1, MAX_TRIES + 1):
        reply = send_to_model(messages)
        data, problems = parse_reply(reply)
        if not problems:
            return data, []
        print(f"  {os.path.basename(path)}: attempt {attempt} failed: {problems}")
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content":
                         "Your reply had these problems: " + "; ".join(problems) +
                         ". Return only the corrected JSON object."})
    return data, problems

## Step 9: Check the numbers

Valid JSON isn't the same as correct data. Like a bank clerk checking a cheque, we check that the numbers on the invoice agree with each other. These checks don't trigger a retry: if the **invoice itself** is wrong, we want to know, not have the AI quietly fix it.

In [ ]:
def close(a, b):
    return abs(a - b) <= TOLERANCE

def check_totals(data):
    warnings = []
    items_sum = 0.0
    for n, item in enumerate(data["line_items"], start=1):
        items_sum += item["amount"]
        if not close(item["quantity"] * item["unit_price"], item["amount"]):
            warnings.append(f"line {n}: quantity x unit price is not the amount")
    if not close(items_sum, data["subtotal"]):
        warnings.append(f"lines add up to {items_sum:,.2f}, subtotal says {data['subtotal']:,.2f}")
    if not close(data["subtotal"] * data["tax_percent"] / 100, data["tax_amount"]):
        warnings.append("tax amount doesn't match tax percent")
    expected_total = data["subtotal"] + data["tax_amount"]
    if not close(expected_total, data["total"]):
        warnings.append(f"subtotal + tax = {expected_total:,.2f}, but total says {data['total']:,.2f}")
    return warnings

## Step 10: Read all the invoices and build the table

In [ ]:
import pandas as pd

summary_rows = []
item_rows = []
needs_person = []

for path in invoice_paths:
    file_name = os.path.basename(path)
    print("Reading", file_name)
    data, problems = read_invoice(path)
    if problems:
        needs_person.append((file_name, problems))
        continue
    warnings = check_totals(data)
    summary_rows.append({
        "file": file_name,
        "vendor": data["vendor"],
        "invoice_number": data["invoice_number"],
        "invoice_date": data["invoice_date"],
        "subtotal": data["subtotal"],
        "tax_percent": data["tax_percent"],
        "tax_amount": data["tax_amount"],
        "total": data["total"],
        "checks": "; ".join(warnings) if warnings else "ok",
    })
    for item in data["line_items"]:
        item_rows.append({"file": file_name, **item})

invoices = pd.DataFrame(summary_rows)
line_items = pd.DataFrame(item_rows)

print("\nInvoices:")
display(invoices)
print("Line items:")
display(line_items)

if needs_person:
    print("Send these to a person to check:")
    for file_name, problems in needs_person:
        print(" -", file_name, problems)

## Step 11: Use the table

Now it's ordinary data. Add things up, sort, filter, and save.

In [ ]:
print(f"Invoices read: {len(invoices)}  |  total payable: {invoices['total'].sum():,.2f}")

problem_invoices = invoices[invoices["checks"] != "ok"]
print(f"Invoices with a warning: {len(problem_invoices)}")
display(problem_invoices[["vendor", "invoice_number", "total", "checks"]])

invoices.to_csv("invoices.csv", index=False)
line_items.to_csv("line_items.csv", index=False)
print("Saved invoices.csv and line_items.csv. Open the folder icon on the left to download them.")